<a href="https://colab.research.google.com/github/sunonmountain/Revenue-Integrity-Engine/blob/main/05_Autonomous_High_Ticket_Outreach_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install -U google-genai pandas pydantic

import pandas as pd
import json
import time
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from google.colab import userdata

# --- 1. CONFIGURATION ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"

# UPDATED: Using the most standard, high-compatibility strings
PRIMARY_MODEL = "gemini-3-flash-preview"
FALLBACK_MODEL = "gemini-3.1-flash-lite-preview" # Standardized naming to fix 404

# --- 2. DATA MODELS & ROI ---
class OutreachDraft(BaseModel):
    subject_line: str = Field(description="Curiosity-inducing subject line.")
    email_body: str = Field(description="Direct, personalized email body.")
    ai_strategy: str = Field(description="The psychological angle used.")

class OutreachROI:
    def __init__(self):
        self.count_emails_drafted = 0
        self.start_time = time.time()
        self.human_time_saved_hours = 0.0
        self.hourly_rate_gbp = 25.0

    def log_action(self):
        self.count_emails_drafted += 1
        self.human_time_saved_hours += 0.25

    def get_report(self):
        duration = round(time.time() - self.start_time, 2)
        savings = round(self.human_time_saved_hours * self.hourly_rate_gbp, 2)
        return {
            "Total_Emails_Drafted": self.count_emails_drafted,
            "Execution_Time_Seconds": duration,
            "Human_Time_Saved_Hours": round(self.human_time_saved_hours, 2),
            "Est_Labour_Savings_GBP": savings,
            "Infrastructure_Cost_GBP": 0.00,
            "Net_ROI_GBP": savings
        }

# --- 3. SANITIZATION ---
def normalize_text(text):
    if not isinstance(text, str): return text
    replacements = {'â€”': '—', 'â€': '—', 'â€™': "'", '\u2014': '-', '\u2019': "'"}
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text.strip()

# --- 4. THE OUTREACH ENGINE ---
class HighTicketOutreachEngine:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)
        self.roi = OutreachROI()

    def generate_with_failover(self, first_name, company, trigger, hook):
        system_instruction = (
            "You are an Elite B2B Copywriter. Write a 1-to-1 email that is brief, "
            "direct, and uses a peer-to-peer tone. Rules: 1. NO 'I hope this finds you well.' "
            "2. Lead with the Research Hook. 3. Soft CTA."
        )
        prompt = f"Write to {first_name} at {company}. Context: {trigger}. Hook: {hook}."

        try:
            # ATTEMPT 1: Primary (Gemini 3)
            return self._api_call(PRIMARY_MODEL, system_instruction, prompt, use_thinking=True)
        except Exception as e:
            # If primary fails (Quota or 404), try stable fallback
            print(f"   ⚠️ Primary issue: {str(e)[:50]}. Falling back to {FALLBACK_MODEL}...")
            return self._api_call(FALLBACK_MODEL, system_instruction, prompt, use_thinking=False)

    def _api_call(self, model_id, system_instruction, prompt, use_thinking=False):
        config_params = {
            "system_instruction": system_instruction,
            "response_mime_type": "application/json",
            "response_schema": OutreachDraft
        }

        # Thinking logic only for Gemini 3
        if use_thinking:
            config_params["thinking_config"] = types.ThinkingConfig(thinking_level="LOW")

        response = self.client.models.generate_content(
            model=model_id,
            contents=prompt,
            config=types.GenerateContentConfig(**config_params)
        )
        self.roi.log_action()
        return json.loads(response.text)

# --- 5. EXECUTION ---
def run_week_5(demo_mode=True):
    input_file = 'Week4_Gold_Intent_Data.csv'
    if not os.path.exists(input_file):
        print("❌ Week 4 CSV not found.")
        return

    df = pd.read_csv(input_file)
    engine = HighTicketOutreachEngine(GEMINI_API_KEY)
    eligible = df[df['Outreach_Hook'].notna()]
    targets = eligible.head(5) if demo_mode else eligible

    print(f"🚀 Processing {len(targets)} drafts...")

    for index, row in targets.iterrows():
        try:
            draft = engine.generate_with_failover(
                row['First_Name'], row['Company'], row['Trigger_Event'], row['Outreach_Hook']
            )
            df.at[index, 'Email_Subject'] = normalize_text(draft['subject_line'])
            df.at[index, 'Email_Body'] = normalize_text(draft['email_body'])
            df.at[index, 'AI_Strategy_Logic'] = normalize_text(draft['ai_strategy'])
            print(f"   ✅ Success: {row['First_Name']} @ {row['Company']}")
            time.sleep(1.2)
        except Exception as e:
            print(f"   ❌ Final Failure for {row['Company']}: {str(e)[:100]}")

    df.to_csv('Week5_Final_Outreach_Leads.csv', index=False, encoding='utf-8-sig')
    print("\n" + "="*35 + "\nWEEK 5 ROI REPORT\n" + json.dumps(engine.roi.get_report(), indent=4) + "\n" + "="*35)

run_week_5(demo_mode=True)

🚀 Processing 5 drafts...
   ⚠️ Primary issue: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm. Falling back to gemini-3.1-flash-lite-preview...
   ✅ Success: Gavin @ Techflow Solutions
   ⚠️ Primary issue: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm. Falling back to gemini-3.1-flash-lite-preview...
   ✅ Success: Gavin @ Techflow Solutions
   ⚠️ Primary issue: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm. Falling back to gemini-3.1-flash-lite-preview...
   ✅ Success: Sarah @ Cloudbridge Uk
   ⚠️ Primary issue: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm. Falling back to gemini-3.1-flash-lite-preview...
   ✅ Success: Alistair @ Finscale Ltd
   ⚠️ Primary issue: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm. Falling back to gemini-3.1-flash-lite-preview...
   ✅ Success: Priya @ Greenmetrics

WEEK 5 ROI REPORT
{
    "Total_Emails_Drafted": 5,
    "Execution_Time_Seconds": 37.58,
    "Human_Time_Saved_Hours": 1.25,
    "Est_Labour_Savings_GBP": 31.25,
    "Infras

In [2]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)

for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
